# Version 2 : modèle supervisé calibré — inventaire de crues éclair méditerranéennes

Ce notebook construit et évalue la **Version 2** du cahier des charges (*Requirements Specification*, sections 6 et 8) :
un modèle supervisé (régression logistique et gradient boosting) **appris sur un inventaire d'inondations**
tiré d'événements documentés et d'**étiquettes satellites** (Copernicus EMS), avec des probabilités **calibrées**.

La Tunisie n'a qu'une seule cartographie satellite d'inondation (EMSR319, Nabeul, septembre 2018) : trop peu pour entraîner.
L'inventaire ajoute donc des **crues éclair cartographiées de la même façon sous un climat comparable** (Libye, Espagne, France, Grèce, Italie).
Chaque événement est testé **sans jamais avoir été vu à l'entraînement** ; le pli Nabeul est la validation tunisienne.

Métriques de la section 8 : **rappel au niveau Élevé et plus**, **fausses alertes**, **délai d'anticipation (lead time)**,
**calibration comparée à V1**, **utilité du facteur drainage**.

Scripts : `flood_risk/real/ems.py` (inventaire), `flood_risk/features.py`, `flood_risk/v1.py`, `flood_risk/v2.py`, `flood_risk/train.py`.

In [1]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mp
from scipy.stats import mannwhitneyu
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

from flood_risk.schema import RAW_FEATURES, HISTORY_FEATURES, RELATIVE_FEATURES, DRAINAGE_FEATURES, LABEL, LEVELS
from flood_risk.features import engineer, MONOTONIC, drainage_vulnerability
from flood_risk.train import cross_validate, with_forecast_error, budget_recall, ece, NAMES, LEADS_MIN, MAIN_LEAD
from flood_risk.real.ems import EVENTS
from flood_risk import v1, v2

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
OUT, MODELS = Path("reports/med_imerg"), Path("models/med_imerg")
ERA5_RESULTS = Path("reports/med/cv_metrics.csv")   # même protocole avec la pluie ERA5, pour comparaison
OUT.mkdir(parents=True, exist_ok=True); MODELS.mkdir(parents=True, exist_ok=True)
DATA = "data/inventory_med_imerg.csv.gz"
SEED, NEG_SAMPLE = 0, 0.3
LEVELS_MODE = "area"   # niveaux par budget d'alerte dans chaque zone (FR-24)
KINDS = ["v1", "v1_relative", "logistic", "hgb", "hgb_no_drainage", "hgb_terrain"]
COLORS = {"v1": "#94a3b8", "v1_relative": "#a16207", "logistic": "#2563eb", "hgb": "#dc2626", "hgb_no_drainage": "#f59e0b", "hgb_terrain": "#16a34a"}

def draw_flow(labels, title, figsize=(16, 3.2)):
    colors = ["#dbeafe", "#e0f2fe", "#dcfce7", "#fef9c3", "#fde68a", "#fbcfe8", "#e9d5ff", "#fee2e2"]
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off"); ax.set_xlim(0, len(labels) * 2); ax.set_ylim(0, 2)
    for i, lab in enumerate(labels):
        x = i * 2 + 0.1
        ax.add_patch(mp.FancyBboxPatch((x, 0.45), 1.7, 1.3, boxstyle="round,pad=0.02,rounding_size=0.12",
                                       fc=colors[i % len(colors)], ec="#334155"))
        ax.text(x + 0.85, 1.1, lab, ha="center", va="center", fontsize=9)
        if i < len(labels) - 1:
            ax.annotate("", xy=(x + 2.0, 1.1), xytext=(x + 1.75, 1.1),
                        arrowprops=dict(arrowstyle="-|>", lw=1.5, color="#334155"))
    ax.set_title(title, fontsize=12, weight="bold")
    plt.tight_layout(); plt.show()

draw_flow(["Cartes satellite\nCopernicus EMS\n(13 crues)", "Routes OSM\nsegments 150 m", "Terrain DEM 30 m\n+ béton\nWorldCover",
           "Pluie GPM IMERG\n10 km, 30 min", "Étiquette\ninondé = touche\nla zone d'eau", "Contrôle\nqualité",
           "Entraînement\nV2 + calibration", "Test : 1 crue\njamais vue\n(dont Nabeul)"],
          "Schéma du pipeline V2")

## 1. Sources et construction de l'inventaire

| Donnée | Source | Rôle |
|---|---|---|
| Zones inondées | Copernicus EMS Rapid Mapping (couche *observed event*, produits délimitation / gradation) | **étiquette** `flooded` |
| Emprise cartographiée | Copernicus EMS (*area of interest*) | limite de la zone étudiée |
| Routes, cours d'eau, passages | OpenStreetMap (Overpass) | segments de 150 m, distance à l'oued |
| Terrain | Copernicus GLO-30 (ouverture morphologique pour retirer les bâtiments) | altitude relative, pente, dépressions, TWI |
| Surfaces bâties | ESA WorldCover | imperméabilisation |
| Pluie | **NASA GPM IMERG V07** (satellites + pluviomètres, maille 0,1° ≈ 10 km, pas de 30 min), `flood_risk/real/imerg.py` | chaque segment prend la pluie de sa maille, à l'heure la plus pluvieuse de cette maille pendant l'orage |
| Pluie (ancienne version) | Open-Meteo, réanalyse ERA5 (≈ 30 km) | une seule valeur par zone ; gardée pour la comparaison de la section 7 bis |

**Règle d'étiquetage** : un segment est *inondé* s'il touche (tampon de 15 m) un polygone « zone inondée » ou « trace de crue ».
Une ligne = un segment × une crue. Les percentiles (`*_pct`) sont calculés **à l'intérieur de chaque zone** pour comparer des reliefs différents.

In [2]:
inv = pd.read_csv(DATA)
inv["event"] = inv.event_id.map(EVENTS)
print(f"lignes (segment × crue) : {len(inv):,}")
print(f"crues (événements)      : {inv.event_id.nunique()}")
print(f"zones cartographiées    : {inv.aoi.nunique()}")
print(f"segments inondés        : {inv[LABEL].sum():,} ({inv[LABEL].mean():.2%})")
per_event = inv.groupby(["event_id", "event"]).agg(zones=("aoi", "nunique"), segments=(LABEL, "size"),
                                                  inondes=(LABEL, "sum"), pluie_mm=("storm_total_mm", "max"),
                                                  pic_mmh=("rain_eta_mmh", "max"))
per_event["taux"] = (per_event.inondes / per_event.segments).map("{:.1%}".format)
per_event.round(1)

lignes (segment × crue) : 225,865
crues (événements)      : 13
zones cartographiées    : 28
segments inondés        : 24,565 (10.88%)


,,zones,segments,inondes,pluie_mm,pic_mmh,taux
event_id,event,,,,,,
EMSR257,"Mandra flash flood, Attica, Greece, Nov 2017",1,2777,481,138.3,25.4,17.3%
EMSR319,"Nabeul, Tunisia, Sept 2018",4,30718,184,78.2,17.1,0.6%
EMSR323,"Mallorca flash flood, Spain, Oct 2018",3,2558,648,85.3,16.5,25.3%
EMSR324,"Aude flash flood, France, Oct 2018",2,8157,250,88.2,20.1,3.1%
EMSR327,"Andalusia flash flood, Spain, Oct 2018",1,1272,29,29.6,5.4,2.3%
EMSR388,"South-east Spain DANA, Sept 2019",3,64831,7403,196.7,37.4,11.4%
EMSR413,"Western Attica, Greece, Nov 2019",2,7506,918,47.8,12.0,12.2%
EMSR480,"Crete, Greece, Nov 2020",2,5732,793,273.2,20.1,13.8%
EMSR548,"Eastern Sicily medicane, Italy, Oct 2021",2,35495,204,158.5,19.8,0.6%


**Analyse.** L'inventaire contient :

- **225 865 lignes** (segment × crue) ;
- **13 crues** et **28 zones cartographiées**, en Tunisie, Libye, Espagne, France, Grèce et Italie ;
- **24 565 segments inondés (10,9 %)**.

Deux activations n'ont pas donné de zone exploitable : Malaga 2018 et Crète 2026.
La colonne `pluie_mm` est maintenant le **cumul IMERG maximal** de la crue (maille la plus arrosée).

Trois points comptent pour la suite :

- **Le taux d'inondés varie énormément.** Il est de 0,6 % à Nabeul et en Sicile orientale, et de 20 à 27 % à Mallorca, en Libye et à Valence 2024.
  Copernicus trace ses zones autour des secteurs touchés : ce taux dépend donc **du découpage de la carte** autant que de la gravité de la crue.
- **IMERG corrige une partie des erreurs d'ERA5.** Mallorca passe de 15 à 85 mm, Crète 2020 de 58 à 273 mm, Mandra de 38 à 138 mm.
  À Nabeul, IMERG reste faible (78 mm au maximum contre plus de 200 mm mesurés localement le 22 septembre 2018) : une maille de 10 km lisse encore les cellules orageuses les plus intenses.
- Treize crues, c'est encore peu pour un modèle supervisé. Chaque résultat « par crue » repose sur un seul événement et doit être lu comme une indication, pas comme une preuve.

## 2. Contrôle qualité (QC)

In [3]:
F = RAW_FEATURES + RELATIVE_FEATURES
qc = pd.DataFrame({"manquantes": inv[F].isna().sum(), "min": inv[F].min(), "médiane": inv[F].median(),
                   "max": inv[F].max(), "distinctes": inv[F].nunique()})
qc["constante"] = qc.distinctes <= 1
qc.round(2)

,manquantes,min,médiane,max,distinctes,constante
elevation_rel_m,0,0.00,8.87,336.30,179700,False
slope_pct,0,0.00,2.12,165.32,187432,False
twi,0,3.44,8.50,18.73,187602,False
depression_depth_m,0,0.00,0.00,40.86,20727,False
dist_wadi_m,0,0.00,398.00,5000.00,212332,False
impervious_frac,0,0.00,0.46,1.00,590,False
drain_inlet_density,0,0.00,0.00,60.00,38,False
osm_completeness,0,0.50,0.50,0.50,1,True
underpass,0,0.00,0.00,1.00,2,False
ford,0,0.00,0.00,1.00,2,False


In [4]:
# Les variables de pluie sont-elles constantes à l'intérieur d'une zone ?
rain_cols = ["rain_eta_mmh", "rain_1h_mm", "rain_3h_mm", "rain_24h_mm", "dry_spell_days"]
print("valeurs distinctes de pluie par zone (max) :", inv.groupby("aoi")[rain_cols].nunique().max().to_dict())
dup = inv.duplicated(["segment_id", "event_id"]).sum()
print("doublons segment × crue :", dup, "(zones qui se chevauchent dans une même crue)")
# Un segment vu par deux zones d'une même crue est gardé une fois, inondé si l'une des cartes le dit.
inv = inv.sort_values(LABEL).drop_duplicates(["segment_id", "event_id"], keep="last").sort_index()
print(f"après dédoublonnage : {len(inv):,} lignes, {inv[LABEL].sum():,} inondées ({inv[LABEL].mean():.2%})")

valeurs distinctes de pluie par zone (max) : {'rain_eta_mmh': 23, 'rain_1h_mm': 24, 'rain_3h_mm': 24, 'rain_24h_mm': 24, 'dry_spell_days': 1}
doublons segment × crue : 9694 (zones qui se chevauchent dans une même crue)
après dédoublonnage : 216,171 lignes, 23,340 inondées (10.80%)


In [5]:
by_aoi = inv.groupby("aoi").agg(segments=(LABEL, "size"), taux=(LABEL, "mean"), event=("event_id", "first")).sort_values("taux")
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(by_aoi.index, 100 * by_aoi.taux, color=["#dc2626" if e == "EMSR319" else "#64748b" for e in by_aoi.event])
ax.set_yscale("log"); ax.set_ylabel("segments inondés (%) — échelle log")
ax.set_title("Densité d'étiquettes par zone (rouge = Nabeul)")
plt.xticks(rotation=70, fontsize=8); plt.tight_layout(); plt.show()

**Analyse du contrôle qualité.**

- **Aucune valeur manquante.**
- **Variables constantes, donc inutiles ici :**
  - `osm_completeness` est fixée à 0,5 hors du pilote ;
  - `black_spot_count` est nulle partout, faute d'historique de points noirs.
- **`drain_inlet_density` n'a que 38 valeurs distinctes** : OpenStreetMap ne cartographie presque aucun avaloir autour de la Méditerranée.
- **La pluie varie maintenant à l'intérieur des zones.**
  - IMERG donne jusqu'à **23–24 valeurs distinctes par zone** (6 mailles en médiane), contre une seule avec ERA5.
  - Le pic de pluie va de **3,9 à 37,6 mm/h** selon le segment.
  - Seul `dry_spell_days` (jours secs avant l'orage) garde une valeur par zone, car il vient encore de la réanalyse.
- **9 694 doublons** viennent de zones d'une même crue qui se chevauchent.
  - Ils sont fusionnés : un segment est inondé si une des cartes le dit.
  - Il reste **216 171 lignes, dont 23 340 inondées (10,8 %)**.
- **La densité d'étiquettes va de 0 % à plus de 40 % selon la zone** (graphique en échelle logarithmique).
  - Les modèles doivent donc être jugés **crue par crue** (ROC-AUC, rappel au budget), plutôt que sur des probabilités absolues.

## 3. Exploration : qu'est-ce qui distingue un segment inondé ?

In [6]:
cols = ["elevation_rel_m", "elevation_rel_pct", "dist_wadi_m", "dist_wadi_pct", "slope_pct", "twi", "twi_pct",
        "depression_depth_m", "impervious_frac"]
s = inv.groupby(LABEL)[cols].median().T
s.columns = ["médiane non inondé", "médiane inondé"]
s["ROC-AUC seule (1 = parfait)"] = [max(roc_auc_score(inv[LABEL], inv[c]), 1 - roc_auc_score(inv[LABEL], inv[c])) for c in cols]
s.round(3)

,médiane non inondé,médiane inondé,ROC-AUC seule (1 = parfait)
elevation_rel_m,9.942,5.161,0.679
elevation_rel_pct,0.529,0.301,0.665
dist_wadi_m,426.759,273.929,0.586
dist_wadi_pct,0.510,0.430,0.563
slope_pct,2.294,1.175,0.617
twi,8.427,9.002,0.588
twi_pct,0.486,0.588,0.570
depression_depth_m,0.000,0.000,0.566
impervious_frac,0.478,0.435,0.524


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, ["elevation_rel_pct", "dist_wadi_pct", "twi_pct"]):
    data = [inv.loc[(inv.event_id == e) & (inv[LABEL] == k), c] for e in per_event.index.get_level_values(0) for k in (0, 1)]
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, widths=0.6)
    for i, b in enumerate(bp["boxes"]):
        b.set_facecolor("#cbd5e1" if i % 2 == 0 else "#f87171")
    ax.set_xticks(np.arange(1.5, 2 * len(per_event), 2), per_event.index.get_level_values(0), rotation=60, fontsize=8)
    ax.set_title(f"{c} (gris = sec, rouge = inondé)")
plt.tight_layout(); plt.show()

**Analyse.** Comparés aux segments secs, les segments inondés sont :

- **plus bas dans leur zone** (altitude relative médiane 5,2 m contre 9,9 m, percentile 0,30 contre 0,53) ;
- **plus près d'un cours d'eau** (274 m contre 427 m) ;
- **plus plats** (pente 1,2 % contre 2,3 %) ;
- **plus humides topographiquement** (TWI plus élevé).

C'est le mécanisme d'une crue éclair de plaine : l'eau s'étale dans les points bas proches de l'oued.

Prise seule, la meilleure variable est l'**altitude relative** (ROC-AUC 0,68). Viennent ensuite son percentile (0,67) et la pente (0,62).

Les boîtes par crue montrent que ce signal existe dans la plupart des crues, avec deux exceptions nettes : **Mandra (Grèce)** et **Derna (Libye)**.
Dans ces deux cas, un torrent encaissé ou une rupture de barrage a traversé la ville, et l'eau a touché des rues à toutes les altitudes.

Le béton (`impervious_frac`) est **un peu plus faible** chez les inondés (0,44 contre 0,48). En crue éclair, ce sont surtout les terres basses périurbaines qui débordent, pas les centres-villes. C'est l'inverse de l'inondation urbaine de New York.

## 4. Découpage train / test : une crue entière à la fois

Une même crue ne doit **jamais** être à la fois dans l'entraînement et dans le test, sinon le modèle « reconnaît » la zone
et les scores sont gonflés. On utilise donc une validation croisée **leave-one-event-out** : un pli par crue, chacun teste une crue
entière (toutes ses zones) avec un modèle appris sur toutes les autres.

À l'intérieur de chaque pli d'entraînement, la **calibration** et les **seuils de niveaux** sont appris sur des prédictions
hors-pli (elles aussi groupées par crue), donc la crue testée n'influence rien.

In [8]:
events = list(per_event.index.get_level_values(0))
fig, ax = plt.subplots(figsize=(10, 3.6))
for i, test in enumerate(events):
    for j, e in enumerate(events):
        ax.add_patch(plt.Rectangle((j, len(events) - 1 - i), 0.95, 0.9, color="#dc2626" if e == test else "#93c5fd"))
ax.set_xlim(0, len(events)); ax.set_ylim(0, len(events))
ax.set_xticks(np.arange(len(events)) + 0.47, events, rotation=45, fontsize=8)
ax.set_yticks(np.arange(len(events)) + 0.45, [f"pli {len(events) - k}" for k in range(len(events))], fontsize=8)
ax.set_title("Leave-one-event-out : rouge = crue de test, bleu = entraînement")
plt.tight_layout(); plt.show()

## 5. Variables (features)

`engineer()` transforme les colonnes brutes en variables du modèle. Les contraintes **monotones** imposent au gradient boosting
un sens physique (plus de pluie ne peut pas baisser le risque, plus haut au-dessus du fond de vallée ne peut pas l'augmenter).

In [9]:
X_all = engineer(inv)
desc = {
    "elevation_rel_m": "altitude au-dessus du minimum local", "twi": "indice d'humidité topographique",
    "depression_depth_m": "profondeur de cuvette", "impervious_frac": "part de béton (100 m)",
    "drain_inlet_density": "avaloirs OSM / km", "osm_completeness": "qualité OSM (fixée)",
    "underpass": "passage inférieur", "ford": "gué", "black_spot_count": "points noirs connus",
    "rain_eta_mmh": "pluie prévue (mm/h)", "rain_1h_mm": "pluie 1 h", "rain_3h_mm": "pluie 3 h",
    "rain_24h_mm": "pluie 24 h", "dry_spell_days": "jours secs avant", "flood_reports_250m": "signalements < 250 m",
    "flood_reports_365d": "signalements 365 j", "basin_reports_prior": "avaloirs bouchés signalés",
    "elevation_rel_pct": "percentile d'altitude dans la zone", "dist_wadi_pct": "percentile de distance à l'oued",
    "twi_pct": "percentile TWI", "impervious_pct": "percentile béton", "drainage_vuln": "vulnérabilité drainage (FR-08)",
    "rain_excess_ratio": "pluie / seuil absorbé par le drainage", "runoff_index": "ruissellement = pluie × béton",
    "wadi_rain": "pluie × proximité oued", "log_dist_wadi": "log distance oued", "log_slope": "log pente"}
pd.DataFrame({"description": [desc.get(c, "") for c in X_all.columns],
              "monotone": [{1: "↑ risque", -1: "↓ risque", 0: "libre"}[MONOTONIC.get(c, 0)] for c in X_all.columns],
              "distinctes": X_all.nunique().values,
              "groupe drainage": [c in DRAINAGE_FEATURES or c in ("drainage_vuln", "rain_excess_ratio") for c in X_all.columns]},
             index=X_all.columns)

,description,monotone,distinctes,groupe drainage
elevation_rel_m,altitude au-dessus du minimum local,↓ risque,172213,False
twi,indice d'humidité topographique,↑ risque,178059,False
depression_depth_m,profondeur de cuvette,↑ risque,19588,False
impervious_frac,part de béton (100 m),↑ risque,589,False
drain_inlet_density,avaloirs OSM / km,↓ risque,35,True
osm_completeness,qualité OSM (fixée),libre,1,True
underpass,passage inférieur,↑ risque,2,False
ford,gué,↑ risque,2,False
black_spot_count,points noirs connus,↑ risque,1,False
rain_eta_mmh,pluie prévue (mm/h),↑ risque,155,False


**Analyse.** Le modèle reçoit **27 variables**, mais certaines n'apportent rien dans cet inventaire :

- `osm_completeness`, `black_spot_count` et les trois variables d'historique sont constantes ;
- `underpass` et `ford` sont très rares.

Avec IMERG, les variables de pluie (pic, cumuls sur 1 h, 3 h et 24 h, et leurs dérivées) **varient d'un segment à l'autre** : elles prennent environ 150 valeurs distinctes.
Le modèle combine donc deux types de signal :

- le **terrain** (où l'eau s'accumule) ;
- la **pluie locale** (combien il a plu sur cette maille).

Les contraintes monotones empêchent le modèle d'apprendre des relations absurdes, par exemple « plus de pluie, moins de risque ».

## 6. Entraînement et test

Modèles comparés :

| Code | Modèle |
|---|---|
| `v1` | V1 : indice multicritère à poids fixes (référence du cahier des charges) |
| `v1_relative` | règle simple : bas dans la zone + proche de l'oued (percentiles) |
| `logistic` | V2 régression logistique calibrée |
| `hgb` | V2 gradient boosting monotone calibré |
| `hgb_no_drainage` | V2 gradient boosting **sans** les variables drainage (ablation, section 8) |
| `hgb_terrain` | V2 gradient boosting **sans les variables de pluie** : apprend seulement *où* l'eau s'étale dans une zone |

Pourquoi `hgb_terrain` ? Avec la réanalyse ERA5, la pluie était **la même pour tous les segments d'une zone** et le modèle
apprenait mieux sans elle. Avec IMERG, la pluie varie d'une maille de 10 km à l'autre : comparer `hgb` et `hgb_terrain`
mesure maintenant ce que la pluie satellite apporte.

Pour chaque pli : on garde tous les segments inondés et 30 % des secs (correction de prior King & Zeng ensuite),
la pluie d'entraînement reçoit une erreur de prévision d'un délai aléatoire 0–30 min, puis on teste la crue exclue
à des délais de **0, 10, 20 et 30 min**.

In [10]:
COLS = RAW_FEATURES + HISTORY_FEATURES + RELATIVE_FEATURES + [LABEL, "event_id", "segment_id", "aoi", "rain_interpolated"]
df = inv[[c for c in COLS if c in inv.columns]].copy()
floats = df.select_dtypes("float64").columns
df[floats] = df[floats].astype("float32")
res, oof = cross_validate(df, n_folds=df.event_id.nunique(), seed=SEED, neg_sample=NEG_SAMPLE, kinds=KINDS, levels=LEVELS_MODE)
res.to_csv(OUT / "cv_metrics.csv", index=False)
res.shape

fold 1/13 done (1 test events)


fold 2/13 done (1 test events)


fold 3/13 done (1 test events)


fold 4/13 done (1 test events)


fold 5/13 done (1 test events)


fold 6/13 done (1 test events)


... (4 outputs left out)

fold 11/13 done (1 test events)


fold 12/13 done (1 test events)


fold 13/13 done (1 test events)


(624, 17)

## 7. Résultats globaux (crues jamais vues, délai 20 min)

In [11]:
main = res[(res.target == "flooded") & (res.lead == MAIN_LEAD)]
cols = ["roc_auc", "pr_auc", "recall_top1", "recall_top2", "recall_top5", "recall_high", "precision_high", "accuracy_high"]
summary = main.groupby("model")[cols].mean().loc[KINDS]
summary.index = [NAMES[k] for k in KINDS]
summary.round(3)

,roc_auc,pr_auc,recall_top1,recall_top2,recall_top5,recall_high,precision_high,accuracy_high
V1 multi-criteria,0.577,0.144,0.015,0.027,0.067,0.029,0.173,0.875
Relative susceptibility,0.743,0.223,0.050,0.078,0.198,0.081,0.233,0.878
V2 logistic,0.694,0.233,0.037,0.072,0.160,0.073,0.257,0.879
V2 gradient boosting,0.736,0.205,0.038,0.074,0.177,0.099,0.268,0.879
V2 GB without drainage proxies,0.729,0.200,0.036,0.079,0.168,0.099,0.272,0.879
V2 GB terrain only,0.733,0.239,0.056,0.109,0.213,0.114,0.285,0.880


In [12]:
pivot = main.pivot_table(index="events", columns="model", values="roc_auc").loc[events, KINDS]
pivot_pr = main.pivot_table(index="events", columns="model", values="pr_auc").loc[events, KINDS]
base = inv.groupby("event_id")[LABEL].mean().loc[events]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, tab, title in [(axes[0], pivot, "ROC-AUC par crue testée"), (axes[1], pivot_pr.div(base, axis=0), "PR-AUC / taux de base (gain)")]:
    im = ax.imshow(tab.values, cmap="RdYlGn", aspect="auto", vmin=0.3 if "ROC" in title else 0, vmax=1 if "ROC" in title else 4)
    ax.set_xticks(range(len(KINDS)), [NAMES[k] for k in KINDS], rotation=25, fontsize=8, ha="right")
    ax.set_yticks(range(len(events)), [f"{e} {EVENTS[e][:22]}" for e in events], fontsize=8)
    for i in range(tab.shape[0]):
        for j in range(tab.shape[1]):
            ax.text(j, i, f"{tab.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(title); plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

**Analyse des résultats globaux.** Moyenne sur les 13 crues, chacune testée sans avoir été vue, avec un délai de 20 min :

| Modèle | ROC-AUC | PR-AUC | Rappel top 5 % |
|---|---|---|---|
| V1 multicritère | 0,58 | 0,14 | 0,07 |
| Règle relative (bas + proche oued) | **0,74** | 0,22 | 0,20 |
| V2 logistique | 0,69 | 0,23 | 0,16 |
| **V2 gradient boosting (terrain + pluie)** | **0,74** | 0,21 | 0,18 |
| V2 GB sans drainage | 0,73 | 0,20 | 0,17 |
| V2 GB terrain seul | 0,73 | **0,24** | **0,21** |

1. **V2 bat nettement V1** : 0,74 contre 0,58 en ROC-AUC. V1, avec ses poids et seuils fixes, ne se transfère pas d'une région à l'autre.
2. **Le gradient boosting complet atteint le niveau de la règle relative** (0,74). La régression logistique progresse aussi beaucoup avec IMERG (0,69), mais reste derrière.
3. **En moyenne, les trois meilleures approches se valent** : V2 complet, V2 terrain et la règle relative.
   - V2 terrain garde le meilleur PR-AUC et le meilleur rappel au budget.
   - V2 complet est meilleur sur les crues les plus étendues : SE Espagne 0,76 contre 0,65, Valence 2024 0,72 contre 0,65, et **Nabeul 0,80 contre 0,77**. Ce sont les crues où la pluie varie le plus d'une maille à l'autre.
4. **Les résultats varient beaucoup d'une crue à l'autre** (carte de chaleur) : V2 va de 0,32 (Mandra) à 0,93 (Sicile 2021).
   - **Mandra et Derna (Libye)** ne sont prévues par aucun modèle (ROC-AUC ≤ 0,57). Ce sont des crues torrentielles urbaines, pas des débordements de plaine.
   - Pour les prévoir, il faudrait d'autres variables : surface du bassin versant en amont, torrent canalisé sous la ville.

## 7 bis. Pluie ERA5 contre pluie IMERG

Même protocole (13 crues, leave-one-event-out, délai 20 min, niveaux par budget), seule la source de pluie change.
Les modèles qui n'utilisent pas la pluie (`v1_relative`, `hgb_terrain`) servent de témoin : leur score doit rester identique.

In [13]:
era5 = pd.read_csv(ERA5_RESULTS)
era5 = era5[(era5.target == "flooded") & (era5.lead == MAIN_LEAD)]
comp_cols = ["roc_auc", "pr_auc", "recall_top5", "precision_high"]
comp = pd.concat({"ERA5": era5.groupby("model")[comp_cols].mean(), "IMERG": main.groupby("model")[comp_cols].mean()}, axis=1).loc[KINDS]
comp.index = [NAMES[k] for k in KINDS]
display(comp.round(3))
nab_comp = pd.concat({"ERA5": era5[era5.events == "EMSR319"].set_index("model")[comp_cols],
                      "IMERG": main[main.events == "EMSR319"].set_index("model")[comp_cols]}, axis=1).loc[KINDS]
nab_comp.index = [NAMES[k] for k in KINDS]
print("Nabeul 2018 :"); display(nab_comp.round(3))
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(KINDS))
ax.bar(x - 0.2, comp[("ERA5", "roc_auc")], 0.4, color="#94a3b8", label="pluie ERA5 (≈ 30 km)")
ax.bar(x + 0.2, comp[("IMERG", "roc_auc")], 0.4, color="#2563eb", label="pluie IMERG (≈ 10 km)")
ax.set_xticks(x, comp.index, rotation=20, ha="right", fontsize=8); ax.set_ylim(0.4, 0.85)
ax.axhline(0.5, color="k", ls=":", lw=1); ax.set(ylabel="ROC-AUC moyenne (13 crues)", title="Apport de la pluie satellite IMERG")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

ERA5                                    \
                               roc_auc pr_auc recall_top5 precision_high   
V1 multi-criteria                0.503  0.117       0.046          0.147   
Relative susceptibility          0.743  0.223       0.198          0.233   
V2 logistic                      0.614  0.165       0.162          0.199   
V2 gradient boosting             0.687  0.199       0.187          0.257   
V2 GB without drainage proxies   0.696  0.201       0.199          0.266   
V2 GB terrain only               0.733  0.239       0.213          0.285   

                                 IMERG                                    
                               roc_auc pr_auc recall_top5 precision_high  
V1 multi-criteria                0.577  0.144       0.067          0.173  
Relative susceptibility          0.743  0.223       0.198          0.233  
V2 logistic                      0.694  0.233       0.160          0.257  
V2 gradient boosting             0.736  0.205       0.177          0.268  
V2 GB without drainage proxies   0.729  0.200       0.168          0.272  
V2 GB terrain only               0.733  0.239       0.213          0.285

Nabeul 2018 :


ERA5                                    \
                               roc_auc pr_auc recall_top5 precision_high   
V1 multi-criteria                0.516  0.006       0.038          0.007   
Relative susceptibility          0.802  0.030       0.348          0.042   
V2 logistic                      0.663  0.013       0.158          0.038   
V2 gradient boosting             0.739  0.024       0.147          0.078   
V2 GB without drainage proxies   0.729  0.021       0.141          0.069   
V2 GB terrain only               0.774  0.015       0.130          0.064   

                                 IMERG                                    
                               roc_auc pr_auc recall_top5 precision_high  
V1 multi-criteria                0.546  0.007       0.060          0.011  
Relative susceptibility          0.802  0.030       0.348          0.042  
V2 logistic                      0.695  0.011       0.060          0.036  
V2 gradient boosting             0.798  0.018       0.201          0.098  
V2 GB without drainage proxies   0.765  0.017       0.168          0.090  
V2 GB terrain only               0.774  0.015       0.130          0.064

In [14]:
# La pluie IMERG explique-t-elle mieux la gravité des crues (taux d'inondés par zone) ?
from scipy.stats import spearmanr
z = inv.groupby("aoi").agg(taux=(LABEL, "mean"), pic=("rain_eta_mmh", "mean"), total=("storm_total_mm", "mean"),
                           mailles=("imerg_cell", "nunique"))
print(f"mailles IMERG par zone : médiane {z.mailles.median():.0f}, max {z.mailles.max()}")
print(f"Spearman taux d'inondés / pic IMERG   : {spearmanr(z.taux, z.pic)[0]:.2f}")
print(f"Spearman taux d'inondés / total IMERG : {spearmanr(z.taux, z.total)[0]:.2f}")
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(z.pic, 100 * z.taux, color=["#dc2626" if a.startswith("EMSR319") else "#2563eb" for a in z.index])
ax.set(xlabel="pic de pluie IMERG moyen de la zone (mm/h)", ylabel="segments inondés (%)", yscale="log",
       title="Gravité de la crue selon la pluie IMERG (rouge = Nabeul)")
plt.tight_layout(); plt.show()

mailles IMERG par zone : médiane 6, max 24
Spearman taux d'inondés / pic IMERG   : 0.52
Spearman taux d'inondés / total IMERG : 0.34


**Analyse — ce qu'apporte la pluie satellite IMERG.**

- **Les modèles qui utilisent la pluie progressent tous** :
  - V1 passe de 0,50 à 0,58 de ROC-AUC (+0,07) ;
  - V2 logistique de 0,61 à 0,69 (+0,08), et son PR-AUC de 0,165 à 0,233 ;
  - V2 gradient boosting de 0,69 à 0,74 (+0,05).

  Les deux témoins sans pluie (règle relative, V2 terrain) ne bougent pas : **l'amélioration vient bien de la pluie**.
- **Avec ERA5, la pluie dégradait le modèle** : V2 complet faisait moins bien que V2 terrain (0,69 contre 0,73). **Avec IMERG, elle aide** (0,74 contre 0,73).
  La qualité de la donnée pluie était donc le facteur limitant.
- **À Nabeul**, V2 complet passe :
  - de 0,74 à **0,80** de ROC-AUC, au niveau de la règle relative ;
  - de 26 % à **33 %** de rappel au niveau Élevé ;
  - de 7,8 % à **9,8 %** de précision au niveau Élevé, soit 16 fois le hasard.
- **IMERG explique en partie la gravité des crues.** La corrélation de rang entre le taux d'inondés d'une zone et son pic de pluie atteint **0,52**, alors qu'elle était nulle avec ERA5.
  Le cumul total explique moins bien (0,34) : **c'est l'intensité horaire qui déclenche les crues éclair**, plus que le volume.
- **Limite** : avec une maille de 10 km, IMERG ne distingue pas deux rues voisines. À l'intérieur d'une zone, le classement reste porté par le terrain.

## 8. Validation tunisienne : le pli Nabeul (EMSR319)

In [15]:
nab = res[(res.events == "EMSR319") & (res.target == "flooded") & (res.lead == MAIN_LEAD)].set_index("model").loc[KINDS, cols]
nab.index = [NAMES[k] for k in KINDS]
display(nab.round(3))
m = (df.event_id == "EMSR319").to_numpy()
y_n = df.loc[m, LABEL].to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for k in KINDS:
    fpr, tpr, _ = roc_curve(y_n, oof[k][m]); axes[0].plot(fpr, tpr, color=COLORS[k], label=NAMES[k])
    pr, rc, _ = precision_recall_curve(y_n, oof[k][m]); axes[1].plot(rc, pr, color=COLORS[k], label=NAMES[k])
axes[0].plot([0, 1], [0, 1], "k:", lw=1); axes[0].set(xlabel="faux positifs", ylabel="rappel", title="ROC — Nabeul 2018")
axes[1].axhline(y_n.mean(), color="k", ls=":", lw=1, label="hasard")
axes[1].set(xlabel="rappel", ylabel="précision", title="Précision-rappel — Nabeul 2018", ylim=(0, max(0.1, 3 * y_n.mean() * 3)))
axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()

,roc_auc,pr_auc,recall_top1,recall_top2,recall_top5,recall_high,precision_high,accuracy_high
V1 multi-criteria,0.546,0.007,0.016,0.022,0.060,0.038,0.011,0.975
Relative susceptibility,0.802,0.030,0.071,0.114,0.348,0.141,0.042,0.976
V2 logistic,0.695,0.011,0.005,0.022,0.060,0.120,0.036,0.976
V2 gradient boosting,0.798,0.018,0.016,0.054,0.201,0.326,0.098,0.978
V2 GB without drainage proxies,0.765,0.017,0.033,0.092,0.168,0.299,0.090,0.978
V2 GB terrain only,0.774,0.015,0.011,0.027,0.130,0.212,0.064,0.977


**Analyse du pli tunisien** (Nabeul 2018, jamais vu à l'entraînement, pluie IMERG).

| Modèle | ROC-AUC | Rappel top 5 % | Rappel Élevé (2 %) | Précision Élevé |
|---|---|---|---|---|
| V1 | 0,55 | 0,06 | 0,04 | 1,1 % |
| Règle relative | **0,80** | **0,35** | 0,14 | 4,2 % |
| V2 logistique | 0,70 | 0,06 | 0,12 | 3,6 % |
| **V2 gradient boosting** | **0,80** | 0,20 | **0,33** | **9,8 %** |
| V2 GB terrain seul | 0,77 | 0,13 | 0,21 | 6,4 % |

- **Sans aucune donnée tunisienne, V2 classe Nabeul aussi bien que la meilleure règle (ROC-AUC 0,80)**, bien mieux que V1 (0,55).
- **Au niveau d'alerte Élevé (2 % du réseau de chaque zone), V2 est le meilleur.** Il détecte **33 % des segments inondés**, avec une précision de **9,8 %**, soit **16 fois le taux de base** (0,6 %).
  La règle relative en détecte 14 %, et V1 4 %.
- **À 5 % de budget sur tout Nabeul, la règle relative reste devant** : 35 % contre 20 %.
  Le haut du classement de V2 est en partie tiré par le béton, un signal fort en Espagne mais faible dans la plaine agricole du Cap Bon.
- **Attention : la règle relative a été choisie en regardant Nabeul** (voir `v1.relative_susceptibility`). Son score sur Nabeul est donc optimiste, alors que celui de V2 est honnête.

## 9. Métriques de la section 8

### 9.1 Niveaux d'alerte, rappel au niveau Élevé+ et fausses alertes

**Comment les niveaux sont placés.** Un seuil de probabilité unique ne se transfère pas d'une région à l'autre : le taux d'inondés va de 0,6 % (Nabeul)
à 27 % (Derna), et un seuil appris ailleurs mettait **16 % du réseau de Nabeul en alerte Élevé+, mais 0,1 % de celui de Valence**.
Les niveaux sont donc placés **par budget d'alerte dans chaque zone** (FR-24, `levels_by_share`) : dans chaque zone, les segments sont classés
par risque, et les **10 %** les plus risqués sont au moins *Modéré*, les **2 %** au moins *Élevé*, les **0,5 %** *Très élevé*.
Ce placement n'utilise aucune étiquette de la zone testée : il ne demande que les scores du modèle, disponibles en temps réel.

Mesures au niveau Élevé+ : **rappel** (part des inondations alertées), **précision** (part des alertes justes), **fausses alertes** (1 − précision)
et **accuracy** (part des segments bien classés, alerte ou non).

In [16]:
fa = main.pivot_table(index="events", columns="model", values=["recall_high", "precision_high", "alerts_per_1000"]).loc[events]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
w = 0.13
for i, k in enumerate(KINDS):
    axes[0].bar(np.arange(len(events)) + (i - 2.5) * w, fa[("recall_high", k)], w, color=COLORS[k], label=NAMES[k])
    axes[1].bar(np.arange(len(events)) + (i - 2.5) * w, fa[("precision_high", k)], w, color=COLORS[k])
axes[0].set(ylabel="rappel Élevé+", title="Rappel au niveau Élevé+ par crue (2 % du réseau en alerte)")
axes[1].plot(np.arange(len(events)), base.values, "k_", ms=14, mew=2, label="taux d'inondés (hasard)")
axes[1].set(ylabel="précision Élevé+", title="Précision des alertes Élevé+ par crue (trait noir = hasard)")
for ax in axes:
    ax.set_xticks(range(len(events)), events, rotation=45, fontsize=8)
axes[0].legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()
fa_mean = main.groupby("model")[["recall_high", "precision_high", "false_alarm", "accuracy_high", "alerts_per_1000"]].mean().loc[KINDS]
fa_mean.index = [NAMES[k] for k in KINDS]
fa_mean.loc["Référence : jamais d'alerte"] = [0, np.nan, np.nan, 1 - base.mean(), 0]
fa_mean.round(3)

,recall_high,precision_high,false_alarm,accuracy_high,alerts_per_1000
V1 multi-criteria,0.029,0.173,0.827,0.875,19.817
Relative susceptibility,0.081,0.233,0.767,0.878,19.817
V2 logistic,0.073,0.257,0.743,0.879,19.817
V2 gradient boosting,0.099,0.268,0.732,0.879,19.817
V2 GB without drainage proxies,0.099,0.272,0.728,0.879,19.817
V2 GB terrain only,0.114,0.285,0.715,0.880,19.817
Référence : jamais d'alerte,0.000,NaN,NaN,0.888,0.000


**Analyse — niveaux par budget d'alerte** (2 % de chaque zone au niveau Élevé+). Moyenne sur les 13 crues jamais vues :

| Modèle | Rappel Élevé+ | Précision | Fausses alertes | Accuracy |
|---|---|---|---|---|
| V1 | 2,9 % | 17,3 % | 83 % | 87,5 % |
| Règle relative | 8,1 % | 23,3 % | 77 % | 87,8 % |
| V2 logistique | 7,3 % | 25,7 % | 74 % | 87,9 % |
| V2 GB | 9,9 % | 26,8 % | 73 % | 87,9 % |
| V2 GB terrain | **11,4 %** | **28,5 %** | **72 %** | **88,0 %** |
| Jamais d'alerte | 0 % | – | – | 88,8 % |

- **Le volume d'alertes est maîtrisé** : exactement 2 % du réseau de chaque zone. Avec des seuils de probabilité appris ailleurs, on mettait en alerte 65 à 75 % du réseau.
- **Les alertes de V2 sont plus fiables que le hasard et que V1.**
  - La précision de V2 est de 27 à 29 %, contre environ 11 % pour le hasard (taux moyen d'inondés).
  - Une alerte Élevé de V2 est donc juste **2,5 fois plus souvent qu'une alerte au hasard**, et 1,6 fois plus souvent qu'une alerte de V1.
- **Le rappel reste faible (10–11 %)**. Avec 2 % des routes en alerte, on ne peut couvrir qu'une petite part des zones inondées, qui représentent en moyenne 11 % du réseau cartographié.
- **L'accuracy (88 %) ne dépasse pas la référence « jamais d'alerte » (88,8 %).** Ce n'est pas un défaut du modèle.
  Quand 89 % des segments sont secs, toute alerte dont la précision est inférieure à 50 % fait baisser l'accuracy.
  C'est pourquoi la section 8 juge le système sur le **rappel** et les **fausses alertes**, pas sur l'accuracy.
- **Les crues très étendues profitent moins de ce budget fixe.**
  - Derna, Valence 2024 et SE Espagne ont 11 à 27 % d'inondés, et 2 % d'alertes y couvrent mécaniquement peu de segments inondés.
  - À l'inverse, les crues rares en profitent : à Nabeul (0,6 % d'inondés) et en Sicile 2021, V2 détecte **25 à 33 %** des segments inondés.

**Choisir le budget.** Le budget est un choix de la protection civile : plus on alerte de routes, plus on couvre d'inondations, mais plus il y a de fausses alertes.
La courbe ci-dessous montre ce compromis pour chaque budget (le même dans chaque zone), à partir des prédictions sur les crues jamais vues.

In [17]:
area, y = df["aoi"].to_numpy(), df[LABEL].to_numpy()
rows = []
for k in ["v1", "v1_relative", "hgb", "hgb_terrain"]:
    rank = pd.Series(oof[k]).groupby(area).rank(pct=True, ascending=False, method="first").to_numpy()
    for share in [0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30]:
        alert = rank <= share
        for e in events:
            m_e = (df.event_id == e).to_numpy()
            a_e, y_e = alert[m_e], y[m_e]
            tp = (a_e & (y_e == 1)).sum()
            rows.append({"model": k, "budget": share, "event": e, "recall": tp / max(y_e.sum(), 1),
                         "precision": tp / max(a_e.sum(), 1), "accuracy": (a_e == (y_e == 1)).mean()})
curve = pd.DataFrame(rows).groupby(["model", "budget"])[["recall", "precision", "accuracy"]].mean()
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for k in ["v1", "v1_relative", "hgb", "hgb_terrain"]:
    for ax, c in zip(axes, ["recall", "precision", "accuracy"]):
        ax.plot(100 * curve.loc[k].index, curve.loc[k, c], marker="o", color=COLORS[k], label=NAMES[k])
axes[0].plot([0, 30], [0, 0.3], "k:", lw=1, label="hasard")
axes[1].axhline(base.mean(), color="k", ls=":", lw=1)
axes[2].axhline(1 - base.mean(), color="k", ls=":", lw=1, label="jamais d'alerte")
for ax, c in zip(axes, ["rappel", "précision", "accuracy"]):
    ax.set(xlabel="budget : % des segments de chaque zone en alerte", ylabel=c, title=f"{c} selon le budget")
axes[0].legend(fontsize=7); axes[2].legend(fontsize=7); plt.tight_layout(); plt.show()
curve.loc["hgb_terrain"].round(3)

,recall,precision,accuracy
budget,,,
0.005,0.030,0.250,0.886
0.010,0.064,0.294,0.884
0.020,0.114,0.285,0.880
0.050,0.221,0.257,0.864
0.100,0.350,0.231,0.834
0.200,0.501,0.194,0.766
0.300,0.610,0.174,0.693


**Analyse — choix du budget** (courbe calculée pour V2 terrain, le modèle au meilleur rappel moyen).

| Budget par zone | Rappel | Précision | Accuracy |
|---|---|---|---|
| 0,5 % | 3 % | 25 % | 88,6 % |
| 1 % | 6 % | 29 % | 88,4 % |
| 2 % | 11 % | 29 % | 88,0 % |
| 5 % | 22 % | 26 % | 86,4 % |
| 10 % | 35 % | 23 % | 83,4 % |
| 20 % | 50 % | 19 % | 76,6 % |
| 30 % | 61 % | 17 % | 69,3 % |

- **Jusqu'à 10 % de budget, la précision reste stable** (23 à 29 %), puis elle baisse doucement. Le rappel monte presque linéairement.
- **La cible de 70 % de rappel de la section 8 n'est pas réaliste avec ces données.** Il faudrait alerter plus de 30 % du réseau, avec plus de 80 % de fausses alertes.
- **Proposition** : **Élevé = 2 %** et **Modéré = 10 %** de chaque zone (`schema.ALERT_SHARE`).
  - Le niveau Élevé reste rare et environ 2,5 fois plus fiable que le hasard.
  - Le niveau Modéré couvre environ un tiers des inondations.
  - Ces budgets sont un réglage de la protection civile : ils peuvent changer sans réentraîner le modèle.

### 9.2 Recall à budget d'alerte fixe

En pratique, on ne peut alerter qu'une petite part du réseau. Le rappel quand on n'alerte que les 1 %, 2 % ou 5 % de segments
les plus risqués est la mesure la plus parlante pour la protection civile, et elle ne dépend pas du seuil.

In [18]:
budget = main.groupby("model")[["recall_top1", "recall_top2", "recall_top5"]].mean().loc[KINDS]
ax = budget.rename(index=NAMES).T.plot.bar(figsize=(9, 4), color=[COLORS[k] for k in KINDS], rot=0)
for share, x in zip((0.01, 0.02, 0.05), range(3)):
    ax.plot([x - 0.4, x + 0.4], [share, share], "k:", lw=1)
ax.set(ylabel="part des inondations couvertes", title="Rappel à budget fixe (pointillé = hasard)")
ax.set_xticklabels(["top 1 %", "top 2 %", "top 5 %"]); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
budget.rename(index=NAMES).round(3)

,recall_top1,recall_top2,recall_top5
model,,,
V1 multi-criteria,0.015,0.027,0.067
Relative susceptibility,0.050,0.078,0.198
V2 logistic,0.037,0.072,0.160
V2 gradient boosting,0.038,0.074,0.177
V2 GB without drainage proxies,0.036,0.079,0.168
V2 GB terrain only,0.056,0.109,0.213


**Analyse — rappel à budget fixe** (classement sur toute la crue, sans découpage par zone).

- En alertant les **5 % de segments les plus risqués**, la couverture des inondations est de :
  - **21 %** pour V2 terrain ;
  - **20 %** pour la règle relative ;
  - **18 %** pour V2 complet ;
  - **6,7 %** pour V1, à peine mieux que le hasard (5 %).
- Ce classement global favorise les modèles fondés sur le terrain. V2 complet donne plus de poids à la pluie, qui sépare surtout les zones entre elles, alors que ce test classe tous les segments de la crue ensemble.
- **Pour la protection civile**, cela veut dire que V2 sait indiquer quels secteurs surveiller en priorité, mais pas encore désigner précisément les quelques routes à fermer.

### 9.3 Délai d'anticipation (lead time)

La pluie au moment du passage est remplacée par une **prévision** dont l'erreur croît avec le délai (0 → 30 min),
plus grande quand la pluie 15 min est interpolée d'une prévision horaire.

In [19]:
lt = res[res.target == "flooded"].groupby(["model", "lead"])[["pr_auc", "roc_auc", "recall_top2"]].mean()
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for k in KINDS:
    for ax, c in zip(axes, ["roc_auc", "pr_auc", "recall_top2"]):
        ax.plot(LEADS_MIN, lt.loc[k, c], marker="o", color=COLORS[k], label=NAMES[k])
for ax, c in zip(axes, ["ROC-AUC", "PR-AUC", "rappel top 2 %"]):
    ax.set(xlabel="délai de prévision (min)", ylabel=c, title=f"{c} selon le délai")
axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()
lt["pr_auc"].unstack().loc[KINDS].rename(index=NAMES).round(4)

lead,0,10,20,30
model,,,,
V1 multi-criteria,0.1550,0.1516,0.1441,0.1417
Relative susceptibility,0.2235,0.2235,0.2235,0.2235
V2 logistic,0.2381,0.2395,0.2329,0.2263
V2 gradient boosting,0.2072,0.2098,0.2047,0.2031
V2 GB without drainage proxies,0.2004,0.2022,0.2000,0.1970
V2 GB terrain only,0.2388,0.2388,0.2388,0.2388


**Analyse — délai d'anticipation.** Le PR-AUC baisse quand la pluie prévue devient moins précise :

| Modèle | PR-AUC à 0 min | PR-AUC à 30 min | Perte |
|---|---|---|---|
| V2 gradient boosting | 0,207 | 0,203 | ≈ 2 % |
| V2 logistique | 0,238 | 0,226 | ≈ 5 % |
| V1 | 0,155 | 0,142 | ≈ 9 % |

- **V2 gradient boosting est le plus robuste.** Ses contraintes monotones et la part du terrain dans ses décisions le rendent peu sensible à l'erreur de prévision. **Il reste utilisable 30 min à l'avance.**
- **V1 perd le plus**, car il donne un poids fixe de 30 % à la pluie.
- **Les modèles sans pluie** (V2 terrain, règle relative) ne bougent pas.
- **Limite** : l'erreur de prévision est **simulée** (bruit log-normal croissant avec le délai). La vraie erreur d'un modèle de prévision (AROME, ECMWF) sur des orages convectifs sera probablement plus forte.

### 9.4 Calibration comparée à V1

Une probabilité de 10 % doit correspondre à 10 % de segments inondés. On compare les probabilités hors-pli (délai 20 min)
à la fréquence observée, avec le **Brier score** et l'**ECE** (erreur de calibration attendue, plus petit = mieux).

In [20]:
y = df[LABEL].to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
rows = []
for k in KINDS:
    p = oof[k]
    edges = np.unique(np.quantile(p, np.linspace(0, 1, 13)))
    idx = np.clip(np.searchsorted(edges, p, side="right") - 1, 0, len(edges) - 2)
    pts = np.array([(p[idx == b].mean(), y[idx == b].mean()) for b in range(len(edges) - 1) if (idx == b).any()])
    for ax in axes:
        ax.plot(*pts.T, marker="o", color=COLORS[k], label=NAMES[k])
    rows.append({"modèle": NAMES[k], "proba moyenne": p.mean(), "fréquence réelle": y.mean(),
                 "Brier": np.mean((p - y) ** 2), "ECE": ece(p, y)})
for ax, lim in zip(axes, [1, 0.3]):
    ax.plot([0, lim], [0, lim], "k--", lw=1); ax.set(xlim=(0, lim), ylim=(0, lim), xlabel="probabilité prédite", ylabel="fréquence observée")
axes[0].set_title("Diagramme de fiabilité (toutes crues)"); axes[1].set_title("Zoom 0–30 %")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(OUT / "reliability.png", dpi=130); plt.show()
calib = pd.DataFrame(rows).set_index("modèle"); calib.round(4)

,proba moyenne,fréquence réelle,Brier,ECE
modèle,,,,
V1 multi-criteria,0.1234,0.108,0.1000,0.0549
Relative susceptibility,0.1129,0.108,0.0977,0.0422
V2 logistic,0.1558,0.108,0.1176,0.1428
V2 gradient boosting,0.1059,0.108,0.0994,0.0910
V2 GB without drainage proxies,0.1102,0.108,0.0999,0.0875
V2 GB terrain only,0.1072,0.108,0.1007,0.0780


In [21]:
# Calibration crue par crue : moyenne prédite contre taux réel de la crue testée
cal_ev = pd.DataFrame({k: pd.Series(oof[k]).groupby(df.event_id.values).mean() for k in KINDS}).loc[events]
cal_ev.insert(0, "taux réel", base.values)
cal_ev.rename(columns=NAMES).round(3)

,taux réel,V1 multi-criteria,Relative susceptibility,V2 logistic,V2 gradient boosting,V2 GB without drainage proxies,V2 GB terrain only
EMSR257,0.173,0.151,0.111,0.107,0.109,0.126,0.121
EMSR319,0.006,0.101,0.131,0.126,0.120,0.124,0.136
EMSR323,0.220,0.088,0.111,0.106,0.096,0.094,0.100
EMSR324,0.032,0.117,0.114,0.111,0.102,0.102,0.110
EMSR327,0.023,0.051,0.113,0.109,0.099,0.099,0.121
EMSR388,0.107,0.209,0.110,0.281,0.126,0.141,0.110
EMSR413,0.178,0.064,0.110,0.103,0.096,0.092,0.101
EMSR480,0.138,0.107,0.112,0.107,0.102,0.103,0.103
EMSR548,0.006,0.094,0.134,0.125,0.118,0.118,0.123
EMSR569,0.067,0.093,0.114,0.107,0.099,0.101,0.109


**Analyse — calibration comparée à V1.**

- **Calibration globale de V2 gradient boosting.**
  - Sa probabilité moyenne (0,106) correspond au taux réel (0,108).
  - Son Brier score (0,0994) est légèrement meilleur que celui de V1 (0,1000).
- **L'ECE est en revanche moins bonne que celle de V1** (0,091 contre 0,055) : **la section 8 n'est donc que partiellement satisfaite**.
  - Avec IMERG, V1 est devenu mieux calibré par hasard : son score moyen est proche du taux moyen.
  - La régression logistique est mal calibrée (ECE 0,143) : elle surestime fortement la crue SE Espagne (0,28 prédit contre 0,11 réel).
- **Le tableau par crue montre la vraie limite.** V2 prédit **0,07–0,13 pour presque toutes les crues**, que le taux réel soit de 0,6 % (Nabeul, Sicile 2021) ou de 27 % (Derna).
  - Le modèle sait classer les segments *à l'intérieur* d'une crue, mais pas encore dire *combien* une crue va inonder.
  - Le taux d'inondés dépend largement du découpage des cartes Copernicus, et aucune variable ne peut l'expliquer.
- **Pour la Tunisie**, les probabilités de V2 sont **à recalibrer localement** dès qu'un deuxième événement tunisien sera disponible.
  - En attendant, elles doivent être présentées comme un **classement relatif**, ce que font les niveaux par budget, et non comme une probabilité absolue.

### 9.5 Utilité du facteur drainage

Deux tests : (1) **ablation** — le gradient boosting avec et sans les variables drainage (`drain_inlet_density`, `dry_spell_days`,
`osm_completeness`, `basin_reports_prior`, et les variables dérivées) ; (2) **test de Mann-Whitney** — parmi les segments inondés,
ceux à forte vulnérabilité drainage sont-ils inondés sous une pluie plus faible ?

In [22]:
abl = main.pivot_table(index="events", columns="model", values=["pr_auc", "roc_auc"]).loc[events]
d = pd.DataFrame({"ROC avec drainage": abl[("roc_auc", "hgb")], "ROC sans drainage": abl[("roc_auc", "hgb_no_drainage")],
                  "PR avec drainage": abl[("pr_auc", "hgb")], "PR sans drainage": abl[("pr_auc", "hgb_no_drainage")]})
d["écart PR"] = d["PR avec drainage"] - d["PR sans drainage"]
display(d.round(4))
print("écart moyen de PR-AUC (avec − sans) :", round(d["écart PR"].mean(), 4),
      "| crues où le drainage aide :", int((d["écart PR"] > 0).sum()), "/", len(d))
print("\nvariables drainage disponibles dans l'inventaire :")
print(inv[["drain_inlet_density", "dry_spell_days", "osm_completeness"]].agg(["nunique", "min", "max"]).T)
flooded = inv[inv[LABEL] == 1]
dv = drainage_vulnerability(flooded)
hi, lo = flooded[dv >= dv.quantile(2 / 3)], flooded[dv <= dv.quantile(1 / 3)]
t = mannwhitneyu(hi.rain_eta_mmh, lo.rain_eta_mmh, alternative="less")
print(f"\nMann-Whitney : pluie médiane à l'inondation, forte vulnérabilité {hi.rain_eta_mmh.median():.1f} mm/h "
      f"contre faible {lo.rain_eta_mmh.median():.1f} mm/h, p = {t.pvalue:.3g}")

,ROC avec drainage,ROC sans drainage,PR avec drainage,PR sans drainage,écart PR
events,,,,,
EMSR257,0.3238,0.3195,0.1200,0.1193,0.0006
EMSR319,0.7978,0.7645,0.0178,0.0169,0.0010
EMSR323,0.5947,0.6119,0.2584,0.2716,-0.0132
EMSR324,0.8881,0.8946,0.2097,0.2123,-0.0027
EMSR327,0.8712,0.8681,0.1339,0.1152,0.0187
EMSR388,0.7592,0.7542,0.2132,0.2147,-0.0015
EMSR413,0.7178,0.7230,0.4258,0.4231,0.0027
EMSR480,0.8034,0.7844,0.3023,0.2823,0.0200
EMSR548,0.9317,0.9307,0.0667,0.0586,0.0081


écart moyen de PR-AUC (avec − sans) : 0.0047 | crues où le drainage aide : 9 / 13

variables drainage disponibles dans l'inventaire :
                     nunique  min   max
drain_inlet_density     38.0  0.0  60.0
dry_spell_days           8.0  1.0  60.0
osm_completeness         1.0  0.5   0.5

Mann-Whitney : pluie médiane à l'inondation, forte vulnérabilité 15.9 mm/h contre faible 18.1 mm/h, p = 4.14e-53


**Analyse — utilité du facteur drainage (section 8).**

- **Ablation.** Avec IMERG, garder les variables drainage améliore légèrement le PR-AUC moyen (+0,005). Elles aident sur **9 crues sur 13**.
  À Nabeul, le ROC-AUC passe de 0,765 sans drainage à 0,798 avec, et le rappel au niveau Élevé de 30 % à 33 %.
- **Mais l'effet est faible, et ces variables ne mesurent pas vraiment le drainage ici.**
  - Les avaloirs OSM sont rarissimes et `osm_completeness` est constante.
  - Le gain vient en pratique des **jours secs avant l'orage** (`dry_spell_days`, 5ᵉ variable la plus importante) et du **béton**, qui entrent dans `drainage_vuln`.
  - Un sol sec et compacté ruisselle davantage : c'est un effet d'**état du sol**, pas d'avaloirs bouchés.
- **Test de Mann-Whitney.** Les segments inondés à forte vulnérabilité l'ont été sous une pluie plus faible (15,9 contre 18,1 mm/h, p ≈ 10⁻⁵³).
  Ce résultat va dans le sens de l'hypothèse FR-09, mais il est **confondu** avec le béton et la zone. Il ne prouve pas l'effet des avaloirs.
- **Conclusion : le facteur drainage, tel qu'il est mesuré, apporte un petit gain mais n'est pas démontré.**
  - Pour conclure, il faut des données d'avaloirs réelles, comme le cadastre de l'ONAS ou les signalements « avaloir bouché » de l'application (FR-21).
  - À New York, où ces signalements existaient, ils aidaient le modèle, mais pas la densité d'avaloirs.

## 10. Modèle final et importance des variables

In [23]:
rng = np.random.default_rng(SEED + 1)
train_all = with_forecast_error(df, rng.uniform(0, 30, len(df)), rng)
finals = {k: v2.fit(k, train_all, SEED, neg_sample=NEG_SAMPLE, levels=LEVELS_MODE) for k in ["hgb", "hgb_terrain", "logistic"]}
for k, b in finals.items():
    joblib.dump(b, MODELS / f"v2_{k}.joblib")
pd.DataFrame({NAMES[k]: b.thresholds for k, b in finals.items()}).T.round(4)

,Moderate,High,Very high
V2 gradient boosting,0.1225,0.1935,0.3116
V2 GB terrain only,0.1324,0.1324,0.1324
V2 logistic,0.1070,0.1173,0.1427


In [24]:
BEST = "hgb"   # modèle complet : terrain + pluie IMERG
final = finals[BEST]
X = v2.features(BEST, df)[final.columns]
sample = np.random.default_rng(0).choice(len(X), 20000, replace=False)
imp = permutation_importance(final.model, X.iloc[sample], y[sample], scoring="average_precision", n_repeats=5, random_state=0)
order = np.argsort(imp.importances_mean)[-15:]
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.barh(np.array(final.columns)[order], imp.importances_mean[order], xerr=imp.importances_std[order], color="#dc2626")
ax.set(xlabel="baisse de PR-AUC quand la variable est mélangée", title=f"Importance des variables — {NAMES[BEST]}")
plt.tight_layout(); plt.savefig(OUT / "importance.png", dpi=130); plt.show()

**Analyse — importance des variables.** Calculée pour le modèle final, V2 gradient boosting avec terrain et pluie IMERG, sur un échantillon de l'entraînement, donc optimiste.

1. **`rain_24h_mm`** (pluie des 24 h avant le pic) est la variable la plus importante. Elle mesure la **saturation du sol** : une crue éclair survient quand une forte averse tombe sur un sol déjà mouillé.
2. **Le béton** (`impervious_pct`, `impervious_frac`) et l'**altitude relative** viennent ensuite. Ils indiquent *où*, dans la zone, l'eau va s'accumuler.
3. Viennent ensuite **les jours secs avant l'orage**, la **distance à l'oued**, le **percentile d'altitude**, les **cumuls sur 1 h et 3 h** et la **profondeur de cuvette**.
4. Le pic instantané (`rain_eta_mmh`), la vulnérabilité drainage et la pente apportent peu une fois les autres variables connues.

Le modèle combine donc ce qu'attend la physique : **pluie accumulée sur la maille × position basse proche de l'oued**.
Le poids du béton est en partie un **signal régional** : les zones espagnoles cartographiées entourent des villes très bétonnées, alors qu'à Nabeul l'eau a touché surtout la plaine agricole. C'est le point à surveiller lors de la recalibration tunisienne.

## 11. Conclusion

### Ce que montre ce notebook

1. **Un pipeline V2 complet, sur données réelles.**
   - Inventaire satellite Copernicus EMS : 13 crues, 28 zones, 6 pays.
   - Pluie satellite NASA GPM IMERG (10 km, 30 min).
   - Contrôle qualité, variables physiques, validation *leave-one-event-out* et calibration.
   - Niveaux d'alerte par budget, délai d'anticipation, ablation du drainage.
   - Modèle final enregistré dans `models/med_imerg/`.
2. **V2 dépasse nettement V1 sur des crues jamais vues.**
   - ROC-AUC 0,74 contre 0,58.
   - Précision des alertes Élevé de 27 % contre 17 %, pour un hasard à environ 11 %.
   - Il reste utilisable 30 min à l'avance (−2 % de PR-AUC).
3. **Le modèle se transfère à la Tunisie.** Appris sans aucune donnée tunisienne, V2 classe Nabeul 2018 avec une ROC-AUC de **0,80**.
   Au niveau Élevé (2 % du réseau), il détecte **33 % des segments inondés**, avec une précision **16 fois supérieure au hasard**.
4. **La pluie satellite était le facteur limitant.** Remplacer ERA5 par IMERG fait gagner +0,05 à +0,08 de ROC-AUC aux modèles qui utilisent la pluie.
   La variable la plus utile du modèle est la pluie accumulée sur 24 h.
5. **V2 égale la meilleure règle simple** (bas + proche de l'oued), mais ne la dépasse pas en moyenne : 0,74 contre 0,74. Il la dépasse au niveau d'alerte Élevé à Nabeul : 33 % contre 14 % de rappel.

### Limites

- **13 crues seulement, dont une seule tunisienne** : chaque chiffre par crue repose sur un seul événement.
- **Étiquettes satellites** : elles montrent l'eau quelques jours après le pic, avec des manques possibles (nuages, arbres, rues étroites), et non une route coupée à une heure précise. Le taux d'inondés dépend aussi du découpage des zones.
- **Pluie à 10 km** : IMERG sous-estime encore les cellules orageuses (78 mm à Nabeul contre plus de 200 mm mesurés). Le classement à l'intérieur d'une zone repose surtout sur le terrain.
- **Calibration** : les probabilités ne suivent pas la gravité propre à chaque crue, et l'ECE n'est pas meilleure que celle de V1. Il faut recalibrer localement.
- **Crues torrentielles urbaines** (Mandra, Derna) : elles ne sont pas prévues, car le terrain local ne suffit pas.
- **Facteur drainage** : petit gain, mais non démontré sans vraies données d'avaloirs.
- **Rappel** : à un budget raisonnable (2 % en Élevé), environ 11 % en moyenne. La cible de 70 % n'est pas atteignable sans alerter un tiers du réseau.
- **Erreur de prévision simulée**, et non mesurée sur de vraies prévisions météo.

### Ce que le modèle attend pour progresser

1. **Plus de crues tunisiennes cartographiées**, à fournir par la partie données du projet. Le pipeline réentraîne et recalibre automatiquement (`python -m flood_risk.train --levels area`).
2. **Une pluie plus fine** : radar ou pluviomètres de l'INM.
3. **Les signalements de l'application (FR-21)**, pour constituer un historique local et tester vraiment le drainage.